# Importing libraries

In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing import image as keras_image
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

Read Files

In [58]:
!ls ../data/

animal_names.csv  animals  animals.txt	download_data.py


In [78]:
animal_names = pd.read_csv(os.path.join("../data", "animal_names.csv"))
image_folders = os.listdir(os.path.join("../data", "animals", "animals"))
file_locations = os.path.join("../data", "animals/")

try:
    for image_folder in image_folders:
        if image_folder.lower() == animal_names["antelope"]:
            folders = os.listdir(os.path.join(file_locations, image_folders))
        else:
            print(f"{image_folder} is not in the animal names")
except Exception as e:
    print(f"{e} some files are not available")
    pass

# Make a dataframe for animals and folders_locations

files_and_locations = pd.DataFrame({"animal": animal_names["antelope"], "folder": file_locations + image_folder})
files_and_locations.head(5)




The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all(). some files are not available


,animal,folder
0,badger,../data/animals/hippopotamus
1,bat,../data/animals/hippopotamus
2,bear,../data/animals/hippopotamus
3,bee,../data/animals/hippopotamus
4,beetle,../data/animals/hippopotamus


In [ ]:
# Creating a dataframe for animal images and seeing if they are in the csv
# Convert animal categories to DataFrame for easier comparison
df_animal_categories = pd.DataFrame(animal_categories, columns=["Animal Name"])

# Create empty dataframe for results
my_df = pd.DataFrame(columns=["Images Folder", "Animal Name"])

# Add each animal to the dataframe 
for animal in animal_categories:
    new_row = pd.DataFrame({"Images Folder": [animal], "Animal Name": [animal]})
    my_df = pd.concat([my_df, new_row], ignore_index=True)

my_df.tail(20)

In [ ]:
# let's list all the folders in animals
lst_animal_folders = os.listdir(images_folder)
abs_path = None
animal_folder_path = None
record = []
# iterate through folders to check contents if needed
for lst_animal in lst_animal_folders:
    abs_path = os.path.abspath(os.path.join(images_folder, lst_animal))
    animal_folder_path = os.path.join(images_folder, lst_animal)
    num_images = len(os.listdir(animal_folder_path))
    record.append({
        "Images Folder": abs_path,
        "Animal Name": lst_animal,
        "length": num_images
    })
    # print(f"{lst_animal}: {num_images} images, path: {abs_path}")
#update dataframe adding abslute path, animal name and length based on the length of image
full_df = pd.DataFrame(record)
full_df.head(5)

In [ ]:
def show_images(type_of_animal, image_number=0, location=None):
    # If location is provided, use it to find the animal folder path
    if location is not None:
        animal_folder_path = location
    else:
        animal_folder_path = os.path.join(images_folder, f"{type_of_animal}")
    
    if os.path.exists(animal_folder_path):
        animal_images = os.listdir(animal_folder_path)
        if animal_images:
            # Make sure the requested image number is valid
            if image_number < len(animal_images):
                animal_image_path = os.path.join(animal_folder_path, animal_images[image_number])
                try:
                    with open(animal_image_path, 'rb') as f:
                        img_data = f.read()
                    
                    # Convert to numpy array via PIL
                    from PIL import Image
                    import io
                    img_pil = Image.open(io.BytesIO(img_data))
                    img = np.array(img_pil)
                    
                    # Display with matplotlib
                    plt.figure(figsize=(8, 6))
                    plt.imshow(img)
                    plt.title(f"{type_of_animal.capitalize()} Image #{image_number+1}")
                    plt.axis('off')
                    return img
                except Exception as e:
                    print(f"Error opening image: {e}")
                    return None
            else:
                print(f"Image number {image_number} is out of range. Only {len(animal_images)} images available.")
                return None
        else:
            print(f"No images found in the {type_of_animal} folder")
            return None
    else:
        print(f"Folder not found: {animal_folder_path}")
        return None
    
# show_images("dog", 2, "/app/data/animals/animals/dog")
# show_images("hippopotamus", 2, "/app/data/animals/animals/hippopotamus")
show_images("dog", 12)

In [ ]:
image_folder = "/app/data/animals/animals"
image_path_from_dataframe = full_df["Images Folder"]
print(image_path_from_dataframe)
print(os.listdir(image_folder))
model_path = os.path.join("models", "animal_classifier.h5")
os.makedirs(os.path.dirname(model_path), exist_ok=True)
print(image_folder)

In [ ]:

def configure_device():
    """Configure TensorFlow to use GPU if available, otherwise CPU"""
    gpus = tf.config.list_physical_devices('GPU')
    
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            
            # Use first GPU by default
            tf.config.set_visible_devices(gpus[0], 'GPU')
            logical_gpus = tf.config.list_logical_devices('GPU')
            print(f"Using GPU: {len(gpus)} Physical GPU(s), {len(logical_gpus)} Logical GPU(s)")
            return "GPU"
        except RuntimeError as e:
            print(f"GPU configuration error: {e}")
            print("Falling back to CPU")
            return "CPU"
    else:
        print("No GPU found. Using CPU for training")
        return "CPU"

device_type = configure_device()

In [66]:
def create_name_to_files_mapping():
    """
    Create a dictionary mapping animal names to lists of their image files
    """
    name_to_files = {}
    animal_categories = os.listdir(images_folder)
    
    for animal in animal_categories:
        animal_path = os.path.join(images_folder, animal)
        if os.path.isdir(animal_path):
            files = [os.path.join(animal_path, f) for f in os.listdir(animal_path)]
            name_to_files[animal] = files
    
    return name_to_files